In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/ezzeddinesayed52@gmail.com/FMCG-DE_Project/Scripts/Setups/utilities

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","gross_price","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = (
    f"r2://sportsbar@"
    f"eefc59d684bff864c9de01ed688857a0"
    f".r2.cloudflarestorage.com/"
    f"{data_source}/*.csv"
)
base_path

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(5))

In [0]:

df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df.select("month").distinct().show()

In [0]:
df_silver = df.withColumn(
    "date",
    F.coalesce(
        F.try_to_date(F.col("month"), 'yyyy/MM/dd'),
        F.try_to_date(F.col("month"), 'dd/MM/yyyy'),
        F.try_to_date(F.col("month"), 'yyyy-MM-dd'),
        F.try_to_date(F.col("month"), 'dd-MM-yyyy')
        ),
).drop("month")

display(df_silver.limit(10))

In [0]:
df_silver = df_silver.withColumn(
    "product_id",
    F.col("product_id").cast("string")
)
df_silver.printSchema()

In [0]:
df_silver.select("date").distinct().show()

In [0]:
df_silver = df_silver.withColumn("month", F.month("date"))
df_silver.select("month").distinct().show()

In [0]:
df_silver.select(
    [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_silver.columns
    ]
).show()

In [0]:
display(df_silver.limit(20))

In [0]:
## fixing the not available gross price
df_silver = df_silver.withColumn(
    "gross_price",
    F.abs(
        F.coalesce(
            F.expr("try_cast(gross_price as double)"),
            F.lit(0.0)
        )
    )
)

display(df_silver)


In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_silver.alias("s").join(
    df_products.alias("p"),
    F.col("s.product_id") == F.col("p.product_code"),
    how="left"
)

display(df_joined)

In [0]:
df_joined.printSchema()

In [0]:
df_joined = df_joined.select(
    "s.product_id",
    "s.gross_price",
    "s.date",
    "s.month",
    "s.read_timestamp",
    "s.file_name",
    "s.file_size"
)

display(df_joined)

In [0]:
df_joined.write\
    .format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.withColumn(
    "year",F.year("date")
)
df_gold = df_gold.select("product_id","gross_price","year")

display(df_gold.limit(5))

In [0]:
df_gold.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")
df_child_products = spark.sql(f"SELECT * FROM {catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table.alias("target").merge(
    source = df_child_products.alias("source"),
    condition="target.product_code = source.product_id"
).whenMatchedUpdate(
    set= {
            "price_inr": "source.gross_price",
            "year": "source.year",
        }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_id",
        "price_inr": "source.gross_price",
        "year": "source.year",
    }
).execute()